In [1]:
import bs4
import requests
import pandas as pd

In [2]:
req = requests.get('https://www.amazon.in/iPhone-17-Pro-Max-Promotion/dp/B0FQF9ZLD7/ref=sr_1_1_sspa?crid=28G2OZBR04WIF&dib=eyJ2IjoiMSJ9.ykXF1XhAbrjgr7h2nfvPiOgs21vgiukJCGcJj2ZXxALcUyxZv_yKkFCHdiR_IOI85TNDJIHkIdd01MNgvCEWKYxefI7ZcVwDGdBQGUTT9Ci7skUJQlgPnhTFW3H2xa6jqD1EDbBAxhQTSi42o_vmlKiRtDfjsGwZTxdCeieT4AGSEZgRex2cOe1kJmPPJTRhH4Xwga9pJ1CNlIpBIO9FPbXpKhrlq8u8x10N21utgIw.2ty2G86FV4xKjpMOwVKT1hEjkHcJzRK0hZz8MENQKEw&dib_tag=se&keywords=mobile%2Bphone&qid=1775549240&sprefix=mobile%2Bpho%2Caps%2C480&sr=8-1-spons&aref=BxSHE0KbLb&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&th=1')

In [3]:
print(req)

<Response [200]>


In [4]:
soup = bs4.BeautifulSoup(req.text)
soup

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-in"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [5]:
reviews=soup.find_all('div',{"data-hook":"review-collapsed"});
print(len(reviews))
for review in reviews:
    print(review.get_text() + "\n\n")

8

Good product and product delivery also proffesional




I'm one of those guys who would never buy an iphone but i had to cause I'm a web developer & new advancements in my skills required me to buy an iphone to load my apps check how the ui is in realtime & then there was this solid exchange bonus offer with a credit card cashback so i went for it & it's great. The photos, software & battery life is great (it's my secondary phone) i got it for 127k so i think that's decently priced so go for it.




This is my best purchase ever




UI so smooth, as a dove floats on a moonlit tranquil pond...Cameras so sharp, as a Ronin handles his sword...Speakers so thunderous, even the heavens rumble when one booms a chorus.








                    The media could not be loaded.
                



I purchased a brand new iPhone 17 Pro Max from Amazon, and within days of delivery the device started showing a visible red/dead pixel issue with intermittent flickering. The defect is clearly visi

In [6]:
reviews=soup.find('div',{"data-hook":"review-collapsed"}).get_text();
reviews

'\nGood product and product delivery also proffesional\n'

In [7]:
ratings=soup.find('span',{"data-hook":"rating-out-of-text"}).get_text();
print(ratings)

4.5 out of 5


In [8]:
ratings=soup.find_all('i',{"data-hook":"review-star-rating"});
for rating in ratings:
    print(rating.get_text()+'\n''\n')

5.0 out of 5 stars


5.0 out of 5 stars


5.0 out of 5 stars


5.0 out of 5 stars


1.0 out of 5 stars


5.0 out of 5 stars


5.0 out of 5 stars


4.0 out of 5 stars




In [9]:
# Fetching customer names

In [10]:
# Fetching product name

In [11]:
# Fetching Price of the product

In [12]:
# Fetching specification of product

In [13]:
# Fetching questions

In [14]:
import re
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)

In [15]:
def get_soup(page_url):
    response = session.get(page_url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser"), response.text

def first_text(soup, selectors, default="N/A"):
    for selector in selectors:
        if isinstance(selector, tuple):
            tag, attrs = selector
            node = soup.find(tag, attrs)
        else:
            node = soup.select_one(selector)
        if node:
            text = node.get_text(" ", strip=True)
            if text:
                return text
    return default

def all_texts(soup, selector, default=None):
    nodes = soup.select(selector)
    values = [n.get_text(" ", strip=True) for n in nodes if n.get_text(" ", strip=True)]
    return values if values else (default if default is not None else [])

In [16]:
def scrape_amazon_product(product_url, max_reviews=10):
    soup, html = get_soup(product_url)

    # Extract ASIN from the product URL if possible
    asin_match = re.search(r"/dp/([A-Z0-9]{10})", product_url)
    asin = asin_match.group(1) if asin_match else None

    # Product info
    product_title = first_text(soup, [
        "#productTitle",
        ("span", {"id": "productTitle"}),
    ])

    price = first_text(soup, [
        "span.a-price span.a-offscreen",
        "#priceblock_ourprice",
        "#priceblock_dealprice",
        "#price_inside_buybox",
    ])

    average_rating = first_text(soup, [
        "span[data-hook='rating-out-of-text']",
        "span.a-icon-alt",
    ])

    total_reviews = first_text(soup, [
        "#acrCustomerReviewText",
    ])

    # Specifications / features
    feature_bullets = all_texts(soup, "#feature-bullets ul li span")
    product_spec_table = {}

    # Common Amazon spec tables
    for table_id in [
        "productDetails_techSpec_section_1",
        "productDetails_detailBullets_sections1",
        "productDetails_techSpec_section_2",
    ]:
        table = soup.find("table", id=table_id)
        if table:
            for row in table.find_all("tr"):
                th = row.find("th")
                td = row.find("td")
                if th and td:
                    key = th.get_text(" ", strip=True)
                    val = td.get_text(" ", strip=True)
                    if key and val:
                        product_spec_table[key] = val

    # Reviews page
    if asin:
        reviews_url = f"https://www.amazon.in/product-reviews/{asin}/?ie=UTF8&reviewerType=all_reviews"
    else:
        reviews_url = product_url.split("?")[0].rstrip("/") + "/ref=cm_cr_arp_d_viewopt_sr?ie=UTF8&reviewerType=all_reviews"

    reviews_soup, _ = get_soup(reviews_url)

    reviews = []
    review_cards = reviews_soup.select("div[data-hook='review']")
    for card in review_cards[:max_reviews]:
        customer_name = first_text(card, [
            ("span", {"class": "a-profile-name"}),
        ])
        review_rating = first_text(card, [
            "i[data-hook='review-star-rating'] span.a-icon-alt",
            "i[data-hook='cmps-review-star-rating'] span.a-icon-alt",
        ])
        review_title = first_text(card, [
            "a[data-hook='review-title'] span",
            "span[data-hook='review-title']",
        ])
        review_text = first_text(card, [
            "span[data-hook='review-body'] span",
            "span[data-hook='review-body']",
        ])
        review_date = first_text(card, [
            "span[data-hook='review-date']",
        ])
        verified = first_text(card, [
            "span[data-hook='avp-badge']",
            "span[data-hook='review-body'] span.a-size-mini",
        ], default="")

        review_format = first_text(card, [
            "a[data-hook='format-strip']",
            "span[data-hook='format-strip']",
            "span[data-hook='review-format-strip']",
        ], default="")

        reviews.append({
            "customer_name": customer_name,
            "review_rating": review_rating,
            "review_title": review_title,
            "review_text": review_text,
            "review_date": review_date,
            "comment_tags": ", ".join([x for x in [verified, review_format] if x]),
        })

    return {
        "product_title": product_title,
        "price": price,
        "average_rating": average_rating,
        "total_reviews": total_reviews,
        "feature_bullets": feature_bullets,
        "specifications": product_spec_table,
        "reviews": reviews,
        "reviews_url": reviews_url,
    }

In [17]:
product_url = 'https://www.amazon.in/iPhone-17-Pro-Max-Promotion/dp/B0FQF9ZLD7/ref=sr_1_1_sspa?crid=28G2OZBR04WIF&dib=eyJ2IjoiMSJ9.ykXF1XhAbrjgr7h2nfvPiOgs21vgiukJCGcJj2ZXxALcUyxZv_yKkFCHdiR_IOI85TNDJIHkIdd01MNgvCEWKYxefI7ZcVwDGdBQGUTT9Ci7skUJQlgPnhTFW3H2xa6jqD1EDbBAxhQTSi42o_vmlKiRtDfjsGwZTxdCeieT4AGSEZgRex2cOe1kJmPPJTRhH4Xwga9pJ1CNlIpBIO9FPbXpKhrlq8u8x10N21utgIw.2ty2G86FV4xKjpMOwVKT1hEjkHcJzRK0hZz8MENQKEw&dib_tag=se&keywords=mobile%2Bphone&qid=1775549240&sprefix=mobile%2Bpho%2Caps%2C480&sr=8-1-spons&aref=BxSHE0KbLb&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&th=1'
data = scrape_amazon_product(product_url, max_reviews=10)
data

{'product_title': 'Apple iPhone 17 Pro Max 2 TB: 17.42 cm (6.9″) Display with Promotion, A19 Pro Chip, Best Battery Life in Any iPhone Ever, Pro Fusion Camera System, Center Stage Front Camera; Silver',
 'price': '₹2,29,900.00',
 'average_rating': '4.5 out of 5',
 'total_reviews': '(328)',
 'feature_bullets': ['UNIBODY DESIGN. FOR EXCEPTIONAL POWER — Heat-forged aluminium unibody enclosure for the most powerful iPhone ever made.',
  'DURABLE CERAMIC SHIELD. FRONT AND BACK — Ceramic Shield protects the back of iPhone 17 Pro Max, making it 4x more resistant to cracks. And the new Ceramic Shield 2 on the front has 3x better scratch resistance.',
  'THE ULTIMATE PRO CAMERA SYSTEM — With all 48MP rear cameras and 8x optical-quality zoom — the longest zoom ever on an iPhone. It’s the equivalent of 8 pro lenses in your pocket.',
  '18MP CENTER STAGE FRONT CAMERA — Flexible ways to frame your shot. Smarter group selfies, Dual Capture video for simultaneous front and rear recording, and more.',

In [18]:
print("Product Name:", data["product_title"])
print("Price:", data["price"])
print("Average Rating:", data["average_rating"])
print("Total Reviews:", data["total_reviews"])
print("Reviews URL:", data["reviews_url"])

Product Name: Apple iPhone 17 Pro Max 2 TB: 17.42 cm (6.9″) Display with Promotion, A19 Pro Chip, Best Battery Life in Any iPhone Ever, Pro Fusion Camera System, Center Stage Front Camera; Silver
Price: ₹2,29,900.00
Average Rating: 4.5 out of 5
Total Reviews: (328)
Reviews URL: https://www.amazon.in/product-reviews/B0FQF9ZLD7/?ie=UTF8&reviewerType=all_reviews


In [19]:
print("Key Features:")
for item in data["feature_bullets"][:8]:
    print("-", item)

print("\nSpecifications:")
for k, v in data["specifications"].items():
    print(f"{k}: {v}")

Key Features:
- UNIBODY DESIGN. FOR EXCEPTIONAL POWER — Heat-forged aluminium unibody enclosure for the most powerful iPhone ever made.
- DURABLE CERAMIC SHIELD. FRONT AND BACK — Ceramic Shield protects the back of iPhone 17 Pro Max, making it 4x more resistant to cracks. And the new Ceramic Shield 2 on the front has 3x better scratch resistance.
- THE ULTIMATE PRO CAMERA SYSTEM — With all 48MP rear cameras and 8x optical-quality zoom — the longest zoom ever on an iPhone. It’s the equivalent of 8 pro lenses in your pocket.
- 18MP CENTER STAGE FRONT CAMERA — Flexible ways to frame your shot. Smarter group selfies, Dual Capture video for simultaneous front and rear recording, and more.
- A19 PRO CHIP. VAPOUR COOLED. LIGHTNING FAST — A19 Pro is the most powerful iPhone chip yet, delivering up to 40% better sustained performance.
- BEST BATTERY LIFE IN ANY IPHONE — The unibody design creates massive additional battery capacity, for up to 37 hours of video playback. Charge up to 50% in 20 m

In [20]:
reviews_df = pd.DataFrame(data["reviews"])
reviews_df

""


In [21]:
# Save extracted reviews to CSV for your practical/report
reviews_df.to_csv("amazon_reviews_scraped.csv", index=False)
print("Saved as amazon_reviews_scraped.csv")

Saved as amazon_reviews_scraped.csv
